# Multi-Class News Classifier — EDA & Training Notebook

This notebook mirrors the `src/train.py` pipeline in an interactive, exploratory format.
Great for beginners who want to see each step (data loading, cleaning, Bag of Words,
model training, evaluation) run one cell at a time.

> Run `python data/generate_dataset.py` first if `data/news_dataset.csv` doesn't exist yet.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

from src.preprocess import clean_series

pd.set_option('display.max_colwidth', 120)

## 1. Load the data

In [ ]:
df = pd.read_csv('../data/news_dataset.csv')
print(df.shape)
df.head()

## 2. Explore class balance

In [ ]:
counts = df['category'].value_counts()
counts.plot(kind='bar', color='#2e75b6', figsize=(6,4), title='Articles per Category')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 3. Clean the text

In [ ]:
df['clean_text'] = clean_series(df['text'])
df[['text', 'clean_text']].sample(5, random_state=1)

## 4. Train/test split + Bag of Words

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_text'], df['category'], test_size=0.2, random_state=42, stratify=df['category']
)

vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print('Vocabulary size:', len(vectorizer.vocabulary_))
print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)

## 5. Train & compare models

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Linear SVM': LinearSVC(),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f'{name:<22} accuracy: {accuracy_score(y_test, preds):.4f}')

## 6. Evaluate the best model in detail

In [ ]:
best_model = models['Naive Bayes']  # change based on the results above
preds = best_model.predict(X_test)

print(classification_report(y_test, preds))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=ax, cmap='Blues', colorbar=False, xticks_rotation=45)
plt.tight_layout()
plt.show()

## 7. Try your own headline

In [ ]:
from src.preprocess import clean_text

sample = "The finance minister announced new tax reforms ahead of the election."
cleaned = clean_text(sample)
pred = best_model.predict(vectorizer.transform([cleaned]))
print('Predicted category:', pred[0])